# Chess-Coach Agent -- Gemma 4 E4B QLoRA on Kaggle (2xT4) -- v5 FINAL RUN

v5 pure-chess, native Gemma format. Locked-in config; run top to bottom.

## Just run it
1. Settings -> Accelerator -> GPU T4 x2.  2. Internet On.
3. Add-ons -> Secrets -> HF_TOKEN (WRITE scope -- base download + checkpoint push).
4. Accept the Gemma 4 license once.
5. Run top to bottom. Cell 1 is pre-filled -- no edits. Restart if Cell 4 shows a banner.

## What's locked in
- all-linear + rank 16 + grad_accum 16 -- proven v4 config.
- v5 native format -- tokenizer native tool_calls, enable_thinking, no legacy XML.
- GROUND_WEIGHT=5.0, FINAL_PROSE=0.1 -- fact tokens up-weighted, weak language reference.
- LR 1e-4, 1000 steps, seq 1920 -- ~2.4 epochs over 13,341 rows, ~11-13h on 2xT4.
- DDP=True -- accelerate launch 2 procs.

## The gate (Cell 6.6) -- ~10 min, before the 11h train
CPU preflight (tokenizer + corpus gates) then a 20-step DDP smoke train -- BOTH as
subprocesses so the GPU is fully released before Cell 7. Never pushes to CKPT_REPO.

## Watch during train (two signals)
- val_loss dropping by step 100; best/ saves lowest and auto-mirrors to CKPT_REPO.
- Checkpoint every 50 updates; resume automatic via Cell 4.5 Hub pull.

## Stop rule
After any session, run the serve notebook's probe on ckpt-v5/best. Ship when: train-row
OK >= 5 with real names, STRESS >= 9/10, unseen-skills >= 3/3.

## OOM ladder
RANK 16->8 (keep all-linear) -> DDP=False (1 GPU). seq stays 1920. If Cell 7 OOMs right after
the gate, the kernel still holds GPU memory -- Restart kernel, then run Cells 1-7.


In [ ]:
# Cell 1 — config (E4B QLoRA, Kaggle 2×T4 DDP). FINAL aimed run. Pre-filled, run top-to-bottom.
import os

MODEL  = "gemma4_e4b"
ENGINE = "unsloth"
HF_REPO = {
    ("gemma4_e4b", "unsloth"): "unsloth/gemma-4-E4B-it",
    ("gemma4_e2b", "unsloth"): "unsloth/gemma-4-E2B-it",
    ("gemma4_e4b", "cuda"):    "google/gemma-4-E4B-it",
    ("gemma4_e2b", "cuda"):    "google/gemma-4-E2B-it",
}[(MODEL, ENGINE)]
NO_4BIT = False

CHESS_REPO_URL = "https://github.com/RyanDev1st/llm-and-engine.git"
CHESS_BRANCH = "feat/v5-pure-chess"
WORKDIR = "/kaggle/working/llm-and-engine"
OUTPUT = "gemma4_chess_e4b_kaggle"
DATA_STEM = "v5"

MAX_SEQ = 1920
TARGETS = "all-linear"   # required for format emission (proven)
RANK = 16                # proven sufficient; rank 32 only worsens overfit — do NOT bump
GRAD_ACCUM = 16
DDP = True

# v5 PURE-CHESS run (native Gemma format). Locked in from the one-shot preflight (GO):
#  - all-linear + rank 16 + grad_accum 16 (proven). v5 DROPPED FORMAT_WEIGHT: native single-token
#    markers don't collide with the base prior, so no up-weighting needed. Loss masks user turns +
#    tool OUTPUTS and weights FACT tokens (GROUND_WEIGHT) -> fixes the 80%-no-why confabulation at
#    the data level (grounded-why 1.000 in the corpus).
#  - LR 1e-4 (overfit-free on v4). v5 is 5.5x SMALLER (13,271 rows), so MAX_STEPS=1000 is ~2.4 EPOCHS
#    here (vs only 0.44 epoch on v1_2) -> real convergence AND ~11-13h on 2xT4, fits one sitting.
#    ~32k samples seen, same as v4's full run. Watch val (every 100); raising toward ~1245
#    (= full 3 epochs at DDP=2) is optional only if time allows.
LR = 1e-4
MAX_STEPS = 1000
EVAL_EVERY = 100
MAX_VAL = 128
SAVE_EVERY = 50

# FRESH v4 repo — the loss changed again (name-weighting); do NOT reuse v3/v2/v1 checkpoints.
CKPT_REPO = "RyanDev1st/gemma4-chesscoach-ckpt-v5"
RESUME = False          # auto-managed by Cell 4.5
RESUME_DATASET = ""
if CKPT_REPO:
    os.environ["CHESS_CKPT_REPO"] = CKPT_REPO

print("config:", MODEL, ENGINE, "| seq", MAX_SEQ, "| targets", TARGETS, "| rank", RANK,
      "| lr", LR, "| steps", MAX_STEPS, "| ckpt", CKPT_REPO or "(local only)")


In [ ]:
# Cell 2 — GPU preflight. If this fails: Settings (right panel) → Accelerator → GPU T4 ×2,
# then re-run. A CPU kernel has no `nvidia-smi` and cannot train E4B.
import subprocess, shutil, torch
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                          "--format=csv"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi NOT found — no GPU attached to this kernel.")
assert torch.cuda.is_available(), (
    "No GPU — Settings → Accelerator → GPU T4 ×2 (×2 needed for DDP), then re-run. "
    "Batch/commit runs default to CPU; you MUST select the accelerator before running.")
n = torch.cuda.device_count()
print("cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0), "| count", n)
if n < 2:
    print("\nℹ️  Only 1 GPU — set DDP=False in Cell 1, or pick GPU T4 ×2 for the ~2x DDP speedup.")


In [ ]:
# Cell 3 — clone repo (skip LFS weights; we only need code + data)
import os, subprocess

def run(cmd, **kw):
    print(">", " ".join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    run(["git", "clone", "--depth", "1", "--branch", CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(["git", "-C", WORKDIR, "fetch", "origin", CHESS_BRANCH, "--depth", "1"], env=env)
    run(["git", "-C", WORKDIR, "checkout", "-B", CHESS_BRANCH, "FETCH_HEAD"], env=env)
    run(["git", "-C", WORKDIR, "reset", "--hard", "FETCH_HEAD"], env=env)
run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])


In [ ]:
# Cell 4 — deps. Let Unsloth resolve its own current stack (do NOT pin an old
# transformers — Gemma 4 needs a recent one).
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

if ENGINE == "unsloth":
    pip("--upgrade", "unsloth", "unsloth_zoo", "bitsandbytes")
    pip("python-chess")
    print("unsloth installed. If Kaggle shows a RESTART banner, restart, then run Cells 5+. "
          "If Gemma 4 fails to load with a model-type error, run: pip install -U transformers")
else:
    pip("-U", "transformers", "accelerate", "peft", "trl",
        "bitsandbytes", "datasets", "sentencepiece", "protobuf", "python-chess")
    import transformers, peft, bitsandbytes
    print("transformers", transformers.__version__, "| peft", peft.__version__,
          "| bnb", bitsandbytes.__version__)


In [ ]:
# Cell 4.5 — HF login + AUTO-RESUME pull + CONFIRM (runs BEFORE the base-model download
# and fit tests). Catches a bad/empty checkpoint in ~30s instead of after minutes of
# weight loading. Pulls checkpoint/ (latest, resume) + best/ (lowest-val, serve) from
# CKPT_REPO and SETS `RESUME` from what actually lands on disk. Re-run the notebook each
# session; this continues automatically — no flag, no download, no Kaggle Dataset.
import os, glob, shutil, math
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))   # also authenticates Cell 5 base download

dst = f"{WORKDIR}/runs/{OUTPUT}"
os.makedirs(dst, exist_ok=True)

def _has_state(d):
    return os.path.exists(f"{d}/checkpoint/trainer_state.pt")

def _purge_smoke(reason):
    """Wipe a non-real (Cell 6.6 micro-overfit) checkpoint from the Hub AND local so the
    final run never resumes from smoke. Hub delete is best-effort (the first real save
    overwrites the same path anyway)."""
    print(f"⚠️  {reason} — this is a SMOKE/gate adapter, not a real run. Purging (Hub + local), FRESH start.")
    try:
        from huggingface_hub import HfApi
        for sub in ("checkpoint", "best"):
            try:
                HfApi().delete_folder(path_in_repo=sub, repo_id=CKPT_REPO, repo_type="model")
                print("  hub deleted:", sub)
            except Exception as e:
                print("  hub", sub, "skip (", repr(e), ")")
    except Exception as e:
        print("  hub purge skipped (", repr(e), ")")
    for sub in ("checkpoint", "best"):
        shutil.rmtree(f"{dst}/{sub}", ignore_errors=True)

if CKPT_REPO:
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(repo_id=CKPT_REPO, repo_type="model",
                          allow_patterns=["checkpoint/*", "best/*"], local_dir=dst)
        print("Hub pull OK:", CKPT_REPO)
    except Exception as e:
        print("Hub pull skipped (", repr(e), ") — fresh start unless a local checkpoint exists")
elif RESUME:
    # Legacy fallback: stage from a Kaggle Dataset (zip or folder) in RESUME_DATASET.
    src = RESUME_DATASET
    zips = glob.glob(f"{src}/**/*.zip", recursive=True)
    if zips:
        print("unzip", zips[0], "->", dst); shutil.unpack_archive(zips[0], dst)
    elif os.path.isdir(src):
        for sub in ("checkpoint", "best"):
            s = os.path.join(src, sub)
            if not os.path.isdir(s):
                s = next((q for q in glob.glob(f"{src}/**/{sub}", recursive=True)), None)
            if s and os.path.isdir(s):
                shutil.copytree(s, os.path.join(dst, sub), dirs_exist_ok=True)
                print("copied", s, "->", os.path.join(dst, sub))

# CONFIRM before any weight loading: resume only if BOTH the trainer state AND the adapter
# weights are present (exactly what train_unsloth._load_ckpt needs). Report the step.
RESUME = False
ck = f"{dst}/checkpoint"
adapter = os.path.exists(f"{ck}/adapter_model.safetensors") or os.path.exists(f"{ck}/adapter_model.bin")
if _has_state(dst) and adapter:
    import torch
    st = torch.load(f"{ck}/trainer_state.pt", map_location="cpu")
    bv = st.get("best_val")
    # SMOKE GUARD: a real run validates every EVAL_EVERY(=100) steps -> best_val is FINITE by
    # update 100. best_val inf/None => no val ever ran => the Cell 6.6 micro-overfit (max-val 0)
    # that a pre-fix gate pushed. NEVER resume the final run from it.
    if bv is None or not math.isfinite(bv):
        _purge_smoke(f"checkpoint best_val={bv} (never validated)")
        print("   -> FRESH run. RESUME=False.")
    else:
        RESUME = True
        print(f"✅ CONFIRMED resume @ update {st.get('update')}/{MAX_STEPS} "
              f"(epoch {st.get('epoch')}, best_val {bv:.4f}). Safe to load weights / train.")
        print("   checkpoint files:", os.listdir(ck))
        if os.path.isdir(f"{dst}/best"):
            print("   best/ present (serve):", os.listdir(f"{dst}/best"))
elif _has_state(dst) and not adapter:
    raise SystemExit("checkpoint/trainer_state.pt present but adapter weights MISSING — bad "
                     "pull. Fix CKPT_REPO before wasting time on the base-model download.")
else:
    print("No checkpoint — FRESH run (first session). RESUME=False.")

In [ ]:
# Cell 5 — HF login + download base model (token from Kaggle Secrets)
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

login(UserSecretsClient().get_secret("HF_TOKEN"))
dest = f"{WORKDIR}/src/llm/models/{MODEL}"
snapshot_download(repo_id=HF_REPO, local_dir=dest,
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("base model at", dest)
print(sorted(os.listdir(dest)))


In [ ]:
# Cell 6 - data check (must be the REGENERATED grounded corpus; stored gzipped)
import os, gzip
for name in (f"{DATA_STEM}_train.jsonl", f"{DATA_STEM}_val.jsonl"):
    base = f"{WORKDIR}/data/sft/{name}"
    path = base if os.path.exists(base) else base + ".gz"
    if not os.path.exists(path):
        n = 0
    elif path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
    else:
        n = sum(1 for _ in open(path, encoding="utf-8"))
    print(name, "rows=", n, "(", os.path.basename(path), ")")
    assert n > 0, f"missing/empty {path} - commit the regenerated corpus to the branch first"


In [ ]:
# Cell 6.6 -- CPU preflight + 20-STEP SMOKE (free, ~10 min). The preflight runs
# retrain_preflight.py against the E4B tokenizer (no GPU). The 20-step smoke
# trains a REAL DDP v5 LoRA as a subprocess so the GPU is fully released for
# Cell 7. NEVER pushes to CKPT_REPO. Both must pass.
import subprocess, sys, os
_env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm', 'CHESS_CKPT_REPO': '',
        'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
        'TORCHDYNAMO_DISABLE': '1', 'UNSLOTH_COMPILE_DISABLE': '1'}

# 1) CPU preflight: validator, tokenizer, corpus gates (~3 min)
cmd = [sys.executable, 'scripts/retrain_preflight.py', '--profile', 'v5',
       '--tok-dir', f'{WORKDIR}/src/llm/models/{MODEL}']
print('>', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR, env=_env)

# 2) 20-step DDP smoke -- real v5 train, local only, never pushes to Hub
smoke = subprocess.run([sys.executable, '-m', 'llm_training.kaggle_v5_one_shot',
        'smoke', '--workdir', WORKDIR], cwd=WORKDIR, env=_env)
assert smoke.returncode == 0, 'SMOKE FAILED -- read the trainer output above'
print()
print('GATE OK: preflight + smoke passed. GPU released -- restart kernel if memory is not clean, then rerun Cells 1-5 + Cell 7')


In [ ]:
# Cell 6.7 -- SMOKE-RESET (safe to run before Cell 7, every session). Checks
# the armed checkpoint: a real run has a FINITE best_val by update 100 (validates
# every EVAL_EVERY=100); a smoke/gate adapter has best_val inf/None (max-val 0).
# REPORT only -- never purge the Hub (Cell 4.5 pulled what was there). If the
# armed checkpoint is smoke/none, set RESUME=False so Cell 7 trains FRESH.
import os, math, torch
_st = f'{WORKDIR}/runs/{OUTPUT}/checkpoint/trainer_state.pt'
_bv = torch.load(_st, map_location='cpu').get('best_val') if os.path.exists(_st) else None
if _bv is not None and math.isfinite(_bv):
    print(f'REAL checkpoint (best_val {_bv:.4f}) -- keeping it; RESUME stays {RESUME}.')
else:
    print(f'best_val={_bv} (smoke/none) -- RESUME=False, Cell 7 trains FRESH from base.')
    RESUME = False


In [ ]:
# Cell 6.8 — OPTIONAL 30-MIN PROBE (run BEFORE Cell 7; skip if out of time). Validates the v5
# enable_thinking bet on a TRAINED adapter: micro-trains a tiny REAL-CONFIG v5 LoRA, then 6.9
# serves it with enable_thinking=True. Single-GPU subprocess so the GPU is fully released for 6.9.
# ~7-12 min here (incl. base load) + ~8 min in 6.9. FREE seq-1920 fit test: OOMs here if 1920
# doesn't fit a T4. Writes runs/v5_probe (separate from the real run's runs/<OUTPUT>).
import subprocess, sys, os
PROBE_STEPS, PROBE_EX = 300, 128       # ~2-3 epochs over 128 rows; raise PROBE_STEPS for a stronger overfit
probe_train = [sys.executable, '-m', 'llm_training.run_train',
    '--model', MODEL, '--engine', ENGINE, '--data-stem', 'v5',
    '--max-examples', str(PROBE_EX), '--max-steps', str(PROBE_STEPS), '--grad-accum', '1',
    '--rank', str(RANK), '--targets', TARGETS, '--lr', str(LR), '--max-seq', str(MAX_SEQ),
    '--max-val', '0', '--eval-every', '100000', '--save-every', str(PROBE_STEPS),
    '--output', 'v5_probe']
print('>', ' '.join(probe_train))
subprocess.run(probe_train, check=True, cwd=WORKDIR,
    # CHESS_CKPT_REPO='' keeps the probe adapter LOCAL — never push it to the v5 Hub repo,
    # or a later Cell 4.5 auto-resume would resume the REAL run from this 300-step overfit.
    env={**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm', 'CHESS_CKPT_REPO': '',
         'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
         'TORCHDYNAMO_DISABLE': '1', 'UNSLOTH_COMPILE_DISABLE': '1'})
print('micro-adapter ->', f'{WORKDIR}/runs/v5_probe/checkpoint')


In [ ]:
# Cell 6.9 — serve the micro-adapter with enable_thinking=True and judge PASS/FAIL on:
#   NATIVE_COT (does the LoRA still think?), ROUTES, GROUNDED, FAST_DIRECT, NO_SOUP.
# Own subprocess => own clean GPU. check=True so a probe FAILURE blocks the kernel.
# GREEN -> launch Cell 7.  RED (CoT suppressed) -> do NOT spend the full run; fix v5 thinking rows first.
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'llm_training.probe_v5_enable_thinking'],
    check=True, cwd=WORKDIR,
    env={**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm',
         'PROBE_BASE': f'{WORKDIR}/src/llm/models/{MODEL}',
         'PROBE_ADAPTER': f'{WORKDIR}/runs/v5_probe/checkpoint'})


In [ ]:
# Cell 7 - TRAIN (E4B QLoRA, Unsloth, all-linear, format-weighted loss). The ONLY cell you
# wait on. DDP=True -> accelerate launch 2 procs (~2x). Checkpoints every SAVE_EVERY + on
# timeout/crash; best/ = lowest val (-> SERVE). With CKPT_REPO set, best/ + checkpoint/ mirror
# to the Hub each save, so a 12h kill can't lose progress and a re-run resumes automatically.
import subprocess, sys, os, torch, math, shutil
# SMOKE GUARD (protects the "skip 6.6, jump to 7" path): never resume the FINAL run from the
# Cell 6.6 micro-overfit adapter. A real run validates every EVAL_EVERY(=100) steps -> finite
# best_val by update 100; best_val inf/None => smoke adapter a pre-fix gate pushed. If the armed
# RESUME points at one, purge it (Hub + local) and train FRESH. Runs even if Cell 4.5 wasn't re-run.
if RESUME:
    _st = f"{WORKDIR}/runs/{OUTPUT}/checkpoint/trainer_state.pt"
    if os.path.exists(_st):
        _bv = torch.load(_st, map_location="cpu").get("best_val")
        if _bv is None or not math.isfinite(_bv):
            print(f"best_val={_bv} (smoke adapter) -- RESUME=False, fresh start.")
            RESUME = False
# GPU-clean guard: each DDP rank loads E4B 4-bit (~9.3 GiB). If this kernel still holds GPU
# memory (e.g. a leftover model), the load OOMs. The gate now subprocesses its probe, but
# guard anyway — fail loud with a fix, not a cryptic OOM 10 min in.
free, total = torch.cuda.mem_get_info(0)
free_gib = free / 1024**3
print(f"GPU0 free: {free_gib:.1f} GiB / {total/1024**3:.1f} GiB")
assert free_gib >= 11.0, (f"Only {free_gib:.1f} GiB free on GPU0 — something holds GPU memory "
                          "in this kernel. RESTART the kernel, then run Cells 1-7 (you can skip "
                          "the Cell 6.6 gate on the rerun — it already proved the fix).")
args = ['--model', MODEL, '--engine', ENGINE, '--max-steps', str(MAX_STEPS),
        '--rank', str(RANK), '--targets', TARGETS, '--grad-accum', str(GRAD_ACCUM),
        '--lr', str(LR), '--max-seq', str(MAX_SEQ), '--eval-every', str(EVAL_EVERY),
        '--max-val', str(MAX_VAL), '--save-every', str(SAVE_EVERY),
        '--data-stem', DATA_STEM, '--output', OUTPUT]
if NO_4BIT: args.append('--no-4bit')
if RESUME:
    _st = f"{WORKDIR}/runs/{OUTPUT}/checkpoint/trainer_state.pt"
    if os.path.exists(_st):
        _bv = torch.load(_st, map_location="cpu").get("best_val")
        if _bv is None or not math.isfinite(_bv):
            print(f"best_val={_bv} (smoke adapter) -- RESUME=False, fresh start.")
            RESUME = False
  args.append('--resume')
if DDP:
    cmd = ['accelerate','launch','--num_processes','2','--multi_gpu','-m','llm_training.run_train'] + args
else:
    cmd = [sys.executable,'-m','llm_training.run_train'] + args
print('>', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm',
                    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'})# Guard: report only. Cell 4.5 already validated the checkpoint. If the
# local checkpoint is smoke (non-finite best_val), set RESUME=False.


In [ ]:
# Cell 8 — package adapter for download (OPTIONAL; run after training OR a timeout).
# Resume is now AUTOMATIC via the Hub (Cell 4.5 pulls+confirms before weight loading), so
# you do NOT need this zip to continue — it's just a convenient local copy of the
# final/best adapter to pull onto the 4060. runs/OUTPUT has: best/ (lowest-val -> SERVE) +
# checkpoint/ (latest, for resume) + the final adapter. With CKPT_REPO set, best/ and
# checkpoint/ are already on the Hub.
import shutil, os
run_dir = f"{WORKDIR}/runs/{OUTPUT}"
print("run dir contents:", os.listdir(run_dir) if os.path.isdir(run_dir) else "MISSING")
out_zip = f"/kaggle/working/{OUTPUT}"
shutil.make_archive(out_zip, "zip", run_dir)
sz = os.path.getsize(out_zip + ".zip") / 1e6
print(f"\n✅ {out_zip}.zip ({sz:.1f} MB) — download from the Output panel (optional).")
print("   SERVE  -> the best/ folder inside (lowest val), or pull best/ from CKPT_REPO.")
print("   RESUME -> AUTOMATIC next session: just re-run the notebook (Cell 4.5 pulls the")
print("             latest checkpoint from CKPT_REPO and confirms it). No upload needed.")
